# N=100, u_max=1280 smoothness/sharpness long Adam GPU sweep

## Mandatory contract for users and AI agents

**This notebook follows `scripts/templates/boilerplate_run.ipynb`, the source of truth for run notebooks.** Keep its cell order, headings, guards, and adapter calls. Do not add, remove, reorder, merge, or redesign sections without the user's explicit permission in the current conversation.

This is a 15 by 15 Cartesian sweep across three decades, centered on smoothness `2.5e-7` and sharpness `2.5e-8`. Each case uses 50 new Fourier starts plus five smaller perturbations of each of the top five stored controls at the center parameters (25 queried starts, 75 total), with a fresh Adam state. Cases are deliberately processed one at a time so one 75-member, million-step batch fits the 16 GB P100. Stable members are removed at learning-rate boundaries and the log reports the counts. This notebook is configured but not launched automatically.

In [ ]:
from ofc.notebook_workflow import RunNotebook

run_name = "N100_u1280_smoothness_sharpness_revised_schedule_adam_gpu"
workflow = RunNotebook(run_name)

## Create the immutable config

Edit the exposed mappings, then activate once. The run explicitly requests `device: gpu`. `resume_optimizer: false` cold-starts every stored control with a fresh Adam count and moments.

In [ ]:
Activated = False

description = "N=100 u_max=1280 three-decade 15x15 smoothness/sharpness revised-schedule Adam GPU sweep."
reuse_existing = False
parameters = {
    "N": 100,
    "t_interval": 4.0,
    "r_bg": -0.008716,
    "u_isbound": True,
    "v_isbound": True,
    "u_max": 1280.0,
    "v_max": 1000.0,
    "slew_limit": 0.05,
    "optimizer": "adam",
    "schedule": [(1_000, 1.0), (30_000, 0.1), (60_000, 0.5)],
    "adam_learning_rate": 0.1,
    "adam_beta1": 0.95,
    "adam_beta2": 0.999,
    "adam_eps": 1e-8,
    "smoothness": [7.905694150420948e-09, 1.2948686698078024e-08, 2.1208572456101802e-08, 3.473738735932843e-08, 5.68961481518697e-08, 9.318984300787349e-08, 1.5263505741463315e-07, 2.5e-07, 4.094734267385159e-07, 6.706739488199311e-07, 1.0984926401901976e-06, 1.7992141825028801e-06, 2.9469215869839663e-06, 4.826744322208124e-06, 7.905694150420947e-06],
    "u_smooth": None,
    "v_smooth": None,
    "sharpness": [7.905694150420948e-10, 1.2948686698078025e-09, 2.1208572456101802e-09, 3.4737387359328433e-09, 5.68961481518697e-09, 9.318984300787349e-09, 1.5263505741463315e-08, 2.5e-08, 4.0947342673851594e-08, 6.706739488199311e-08, 1.0984926401901975e-07, 1.79921418250288e-07, 2.946921586983966e-07, 4.826744322208123e-07, 7.905694150420948e-07],
    "u_sharp": None,
    "v_sharp": None,
    "block_size": 1_000,
    "J_tol": 1e-5,
    "u_tol": 1e-4,
    "v_tol": 1e-4,
    "projected_gradient_tol": 1e-4,
    "projected_gradient_alpha": 1.0,
}
runtime = {
    "initialisations": 50,
    "fourier_num_modes": 5,
    "fourier_rms_amplitude": 0.3,
    "fourier_intensity_fraction": 0.3,
    "use_jit": True,
    "use_x64": True,
    "device": "gpu",
    "concurrent_workers": 1,
    "max_cases_per_batch": 1,
    "max_initialisations_per_batch": 25,
    "max_batch_elapsed_seconds": 18_000,
    "auto_halt": True,
    "database": "results/results.sqlite3",
}
initialization_query = {
    "where": {"status": "complete", "N": 100, "u_max": 1280.0, "smoothness": 2.5e-7, "sharpness": 2.5e-8},
    "limit": 5,
    "order_by": "best_score",
    "descending": True,
    "control_kind": "best",
    "resume_optimizer": False,
    "perturbed": True,
    "perturbation_levels": [0.0005, 0.001, 0.0025, 0.005, 0.01],
}

config_document = workflow.create_config(
    activated=Activated,
    description=description,
    parameters=parameters,
    runtime=runtime,
    initialization_query=initialization_query,
    reuse_existing=reuse_existing,
)

## Run directly on `bar`'s GPU (detached)

This verifies JAX CUDA visibility and starts a detached process that survives notebook, browser, or laptop disconnection.

In [ ]:
Activated = False

queue_id = None
python_executable = None
extra_arguments = []
detached = True
log_path = None

active_queue_id = workflow.run_on_bar_gpu(
    activated=Activated,
    queue_id=queue_id,
    python_executable=python_executable,
    extra_arguments=extra_arguments,
    detached=detached,
    log_path=log_path,
)

## Submit through Slurm (alternative)

In [ ]:
Activated = False

partition = "gpu"
time = "7-00:00:00"
cpus = 2
memory = "16G"
array = True
array_max_concurrent = 1
job_name = None
extra_arguments = []

active_queue_id = workflow.submit_slurm(
    activated=Activated,
    partition=partition,
    time=time,
    cpus=cpus,
    memory=memory,
    array=array,
    array_max_concurrent=array_max_concurrent,
    job_name=job_name,
    extra_arguments=extra_arguments,
)

## Query persisted data

In [ ]:
inherit_config = True
database = None
queue_id = None
config_run_rank = 1
statuses = None
filters = {}
sweep_parameters = ["smoothness", "sharpness"]
require_saved_stage = True
limit = None
order_by = "run_id"
descending = False

query_result = workflow.query(
    inherit_config=inherit_config,
    database=database,
    queue_id=queue_id,
    config_run_rank=config_run_rank,
    statuses=statuses,
    filters=filters,
    sweep_parameters=sweep_parameters,
    require_saved_stage=require_saved_stage,
    limit=limit,
    order_by=order_by,
    descending=descending,
)

## Figure display and saving

In [ ]:
save_figure = None
figure_format = "png"
preview_dpi = 240
save_dpi = 600

## Unified sweep summary

The history median/spread and objective strip use only the runs shown in each row. Dashed vertical lines mark learning-rate changes.


In [ ]:
sweep_parameter = 'smoothness'  # Required when the query selected multiple sweeps; e.g. "u_max".
history_points = 1200
summary_figure = query_result.plot_summary(
    sweep_parameter=sweep_parameter,
    history_points=history_points,
)
workflow.present_figure(
    summary_figure,
    "01_sweep_summary",
    save_figure=save_figure,
    figure_format=figure_format,
    preview_dpi=preview_dpi,
    save_dpi=save_dpi,
)


## Double sweep summary

In [ ]:
separate_sweep_parameter = "smoothness"
colour_sweep_parameter = "sharpness"
history_points = 1200
double_sweep_figure = query_result.plot_double_sweep_summary(separate_sweep_parameter=separate_sweep_parameter, colour_sweep_parameter=colour_sweep_parameter, history_points=history_points)
workflow.present_figure(double_sweep_figure, "02_double_sweep_summary", save_figure=save_figure, figure_format=figure_format, preview_dpi=preview_dpi, save_dpi=save_dpi)

## Triple sweep summary

In [ ]:
row_sweep_parameter = "u_max"
column_sweep_parameter = "smoothness"
colour_sweep_parameter = "sharpness"
history_points = 1200
triple_sweep_figure = query_result.plot_triple_sweep_summary(row_sweep_parameter=row_sweep_parameter, column_sweep_parameter=column_sweep_parameter, colour_sweep_parameter=colour_sweep_parameter, history_points=history_points)
workflow.present_figure(triple_sweep_figure, "03_triple_sweep_summary", save_figure=save_figure, figure_format=figure_format, preview_dpi=preview_dpi, save_dpi=save_dpi)